# ERCOT - Ancillary Price Adders Data Cleaning (post 2025 Dec)
Raw source: `data/raw/ercot/RTM Price Adders 2021-2025/`
Aggregation Script: `src/transformation/parse_rtm_price_adders_2025dec_2026.py`

**Data source:** `data/parquet/ercot/rtm_price_adders_15min_20251205_20260517.parquet`

Schemas: `RTRDPA`, `RTRDPRU`, `RTRDPRD`, `RTRDPRRS`, `RTRDPECRS`, `RTRDPNS`

## 1. Imports

In [10]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path

PROJECT_ROOT = Path('../..').resolve()

PARQUET = PROJECT_ROOT / '01_data' / '1.2_raw_api' / 'rtm_price_adders_15min_20251205_20260517.parquet'
OUT_DIR = PROJECT_ROOT / '01_data' / '3_analysis' / 'price_adders_analysis' / 'post_202512'
OUT_DIR.mkdir(parents=True, exist_ok=True)
con = duckdb.connect()
print(f'Parquet: {PARQUET}')
print(f'Output Directory: {OUT_DIR}')

Parquet: /Users/zyliazhang/Git/RESEARCH-PJM-ERCOT-Price-Volatility/data/parquet/ercot/rtm_price_adders_15min_20251205_20260517.parquet
Output Directory: /Users/zyliazhang/Git/RESEARCH-PJM-ERCOT-Price-Volatility/data/price_adders_analysis/post_202512


## 2. Dataset Validation and Aggregation

In [11]:
schema = con.execute(f"""
    DESCRIBE SELECT * FROM read_parquet('{PARQUET}')
""").df()
print('=== Schema ===')
print(schema.to_string(index=False))

# time range
summary = con.execute(f"""
    SELECT
        COUNT(*)                        AS total_rows,
        MIN(timestamp)                  AS min_ts,
        MAX(timestamp)                  AS max_ts,
        COUNT(DISTINCT timestamp)       AS unique_timestamps
    FROM read_parquet('{PARQUET}')
""").df()
print('\n=== Time Range ===')
print(f"Min: {summary['min_ts'].iloc[0]}, Max: {summary['max_ts'].iloc[0]}")

# preview
overlap = con.execute(f"""
    SELECT * FROM read_parquet('{PARQUET}')
    ORDER BY timestamp
    LIMIT 5
""").df()
print('\n=== Preview data ===')
overlap

=== Schema ===
     column_name column_type null  key default extra
       timestamp   TIMESTAMP  YES None    None  None
RepeatedHourFlag     VARCHAR  YES None    None  None
          RTRDPA      DOUBLE  YES None    None  None
         RTRDPRU      DOUBLE  YES None    None  None
         RTRDPRD      DOUBLE  YES None    None  None
        RTRDPRRS      DOUBLE  YES None    None  None
       RTRDPECRS      DOUBLE  YES None    None  None
         RTRDPNS      DOUBLE  YES None    None  None

=== Time Range ===
Min: 2025-12-05 00:00:00, Max: 2026-05-16 23:45:00

=== Preview data ===


,timestamp,RepeatedHourFlag,RTRDPA,RTRDPRU,RTRDPRD,RTRDPRRS,RTRDPECRS,RTRDPNS
0,2025-12-05 00:00:00,N,0.0,0.43,3.24,0.44,0.67,2.09
1,2025-12-05 00:15:00,N,0.0,0.22,0.70,0.23,0.45,1.80
2,2025-12-05 00:30:00,N,0.0,0.20,0.99,0.20,0.40,1.60
3,2025-12-05 00:45:00,N,0.0,0.18,0.92,0.18,0.36,1.44
4,2025-12-05 01:00:00,N,0.0,0.85,0.45,0.09,0.15,1.01


In [12]:
# aggregate to hourly average price
df_hourly = con.execute(f"""
    SELECT
        DATE_TRUNC('hour', timestamp) AS date,
        AVG(RTRDPA)    AS RTRDPA,
        AVG(RTRDPRU)   AS RTRDPRU,
        AVG(RTRDPRD)   AS RTRDPRD,
        AVG(RTRDPRRS)  AS RTRDPRRS,
        AVG(RTRDPECRS) AS RTRDPECRS,
        AVG(RTRDPNS)   AS RTRDPNS
    FROM read_parquet('{PARQUET}')
    WHERE RepeatedHourFlag = 'N'
    GROUP BY DATE_TRUNC('hour', timestamp)
    ORDER BY date
""").df()
df_hourly.to_csv(f'{OUT_DIR}/post_202512_price_adders_hourly.csv', index=False)
df_hourly

,date,RTRDPA,RTRDPRU,RTRDPRD,RTRDPRRS,RTRDPECRS,RTRDPNS
0,2025-12-05 00:00:00,0.0,0.2575,1.4625,0.2625,0.4700,1.7325
1,2025-12-05 01:00:00,0.0,0.5325,0.6200,0.0550,0.0850,0.8125
2,2025-12-05 02:00:00,0.0,0.1650,0.3575,0.0300,0.0575,0.5150
3,2025-12-05 03:00:00,0.0,0.2425,0.0175,0.0000,0.0000,0.2525
4,2025-12-05 04:00:00,0.0,0.5800,0.5825,0.0100,0.0100,0.4850
...,...,...,...,...,...,...,...
3906,2026-05-16 19:00:00,0.0,0.2600,0.5675,0.0100,0.0225,0.1000
3907,2026-05-16 20:00:00,0.0,0.1575,0.5275,0.0700,0.1450,0.5675
3908,2026-05-16 21:00:00,0.0,0.4725,1.3400,0.6000,1.2450,4.9750
3909,2026-05-16 22:00:00,0.0,0.1775,0.7800,0.1325,0.2575,1.0425


In [7]:
# zero and null counts
print(f'total hourly records: {len(df_hourly)}')
for col in ['RTRDPA', 'RTRDPRU', 'RTRDPRD', 'RTRDPRRS', 'RTRDPECRS', 'RTRDPNS']:
    zero_count = (df_hourly[col] == 0).sum()
    zero_percent = zero_count / len(df_hourly) * 100
    null_count = df_hourly[col].isnull().sum()
    print(f"{col}: Zero Count = {zero_count} ({zero_percent:.1f}%), Null Count = {null_count}")

total hourly records: 3911
RTRDPA: Zero Count = 3296 (84.3%), Null Count = 0
RTRDPRU: Zero Count = 321 (8.2%), Null Count = 0
RTRDPRD: Zero Count = 117 (3.0%), Null Count = 0
RTRDPRRS: Zero Count = 170 (4.3%), Null Count = 0
RTRDPECRS: Zero Count = 151 (3.9%), Null Count = 0
RTRDPNS: Zero Count = 0 (0.0%), Null Count = 0


In [9]:
# convert to binary
df_hourly_binary = df_hourly.copy()
for col in ['RTRDPA', 'RTRDPRU', 'RTRDPRD', 'RTRDPRRS', 'RTRDPECRS', 'RTRDPNS']:
    df_hourly_binary[col] = (df_hourly_binary[col] > 0).astype(int)
df_hourly_binary.to_csv(f'{OUT_DIR}/post_202512_price_adders_hourly_binary.csv', index=False)
df_hourly_binary

,date,RTRDPA,RTRDPRU,RTRDPRD,RTRDPRRS,RTRDPECRS,RTRDPNS
0,2025-12-05 00:00:00,0,1,1,1,1,1
1,2025-12-05 01:00:00,0,1,1,1,1,1
2,2025-12-05 02:00:00,0,1,1,1,1,1
3,2025-12-05 03:00:00,0,1,1,0,0,1
4,2025-12-05 04:00:00,0,1,1,1,1,1
...,...,...,...,...,...,...,...
3906,2026-05-16 19:00:00,0,1,1,1,1,1
3907,2026-05-16 20:00:00,0,1,1,1,1,1
3908,2026-05-16 21:00:00,0,1,1,1,1,1
3909,2026-05-16 22:00:00,0,1,1,1,1,1
